# 01 - Data exploration & preprocessing (C-MAPSS FD001)

FD001: 100 train engines, 100 test engines, **one operating condition**, **one fault mode (HPC degradation)**. Each row = one flight cycle; columns = unit, cycle, 3 operating settings, 21 sensors.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
%matplotlib inline
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.config import Config, SENSOR_NAMES
from src.data import find_data_dir, load_raw, clean, add_rul, select_features, split_units, fit_scaler, apply_scaler, make_windows, make_test_windows
from src import plots
cfg = Config()
print("data folder:", find_data_dir("FD001"))

data folder: /home/claude/cmapss-rul/data


## 1. Load & clean
The raw files end every line with a space -> pandas creates an **empty trailing column**, which we drop.

In [2]:
train, test, rul_true = load_raw("FD001")
print("raw shapes:", train.shape, test.shape, rul_true.shape)
train, test = clean(train), clean(test)
print("missing values:", int(train.isna().sum().sum()), int(test.isna().sum().sum()))
print("engine life (train): min %d / mean %.0f / max %d cycles" % tuple(train.groupby("unit").size().agg(["min","mean","max"])))
print("test engine length: min %d, max %d" % (test.groupby("unit").size().min(), test.groupby("unit").size().max()))
train.head()

raw shapes: (20631, 26) (13096, 26) (100,)
missing values: 0 0
engine life (train): min 128 / mean 206 / max 362 cycles
test engine length: min 31, max 303


,unit,cycle,op1,op2,op3,s1,s2,s3,s4,s5,...,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


## 2. RUL target (capped at 125)
Training RUL = last cycle of the engine - current cycle. Early in life there is no visible degradation, so we cap at **125** (piece-wise linear target).

In [3]:
train = add_rul(train, cfg.rul_cap)
plots.fig_rul_labels(train, cfg.rul_cap); plt.show()
train[["unit","cycle","rul_raw","rul"]].tail(3)

,unit,cycle,rul_raw,rul
20628,100,198,2,2
20629,100,199,1,1
20630,100,200,0,0


## 3. Constant sensors
Sensors/settings with a single value carry no information and would only add noise (and break scaling).

In [4]:
features, const = select_features(train, cfg)
print("constant columns :", const)
print("features kept (%d):" % len(features), features)
plots.fig_constant_sensors(train, const); plt.show()

constant columns : ['op3', 's1', 's5', 's10', 's16', 's18', 's19']
features kept (15): ['s2', 's3', 's4', 's6', 's7', 's8', 's9', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21']


In [5]:
plots.fig_sensor_trends(train, features); plt.show()
plots.fig_corr(train, features); plt.show()

Note `s6` is almost constant (only 21.60/21.61 flicker) - the LSTM learns to ignore it; `op1/op2` are sensor-like noise in FD001, so they are excluded by default (`Config.use_op_settings=False`).

## 4. Engine-level 80/20 split, scaling, windows
Split by **engine** (never by row) so no engine appears in both sets. The scaler is fitted on the training engines **only**.

In [6]:
tr_u, va_u = split_units(train["unit"].unique(), cfg.val_frac, cfg.seed)
tr_df, va_df = train[train.unit.isin(tr_u)], train[train.unit.isin(va_u)]
scaler = fit_scaler(tr_df, features)
tr_s, va_s, te_s = (apply_scaler(x, scaler, features) for x in (tr_df, va_df, test))
print("train engines:", len(tr_u), "| val engines:", len(va_u))
X_tr, y_tr, _ = make_windows(tr_s, features, cfg.window)
X_va, y_va, _ = make_windows(va_s, features, cfg.window)
X_te, te_units = make_test_windows(te_s, features, cfg.window)
print("X_train", X_tr.shape, "| X_val", X_va.shape, "| X_test", X_te.shape)
print("scaled train range: %.2f .. %.2f (val may slightly exceed - that is expected, no leakage)" % (tr_s[features].min().min(), tr_s[features].max().max()))
print("val range        : %.2f .. %.2f" % (va_s[features].min().min(), va_s[features].max().max()))

train engines: 80 | val engines: 20
X_train (14459, 30, 15) | X_val (3272, 30, 15) | X_test (100, 30, 15)
scaled train range: 0.00 .. 1.00 (val may slightly exceed - that is expected, no leakage)
val range        : -0.04 .. 1.06
